In [1]:
# this notebook will normalize the data
# while keeping key level as feature
import pandas as pd
import json
from pathlib import Path


folder_path = Path.cwd().parents[1] / "data" / "mlData"
src_path = folder_path / "raw" / "BTCUSD-5m-v6.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed
0,1704037500000,42497.851562,42518.148438,42480.691406,42482.250000,56.512959,NaN,1704037500000,0,0,-0.000367,NaN,NaN,NaN,1704037500000,1704037500000,0,0
1,1704037800000,42482.238281,42500.000000,42411.101562,42414.421875,47.354488,NaN,1704037800000,0,0,-0.001596,NaN,-0.001597,NaN,1704037500000,1704037500000,0,0
2,1704038100000,42414.429688,42471.988281,42414.421875,42457.171875,43.228668,NaN,1704038100000,0,0,0.001008,NaN,0.001008,NaN,1704037500000,1704037500000,0,0
3,1704038400000,42457.171875,42484.828125,42436.468750,42449.601562,74.418266,NaN,1704038400000,0,0,-0.000178,NaN,-0.000178,NaN,1704038400000,1704037500000,0,0
4,1704038700000,42449.589844,42488.000000,42449.589844,42487.988281,38.818489,NaN,1704038700000,0,0,0.000905,NaN,0.000904,NaN,1704038400000,1704037500000,0,0


In [2]:
print(df.columns)

Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'atr42', 'ts_5m',
       'label', 'train_mask', 'body_pct', 'vol_norm', 'close_ret1',
       'atr42_pct', 'ts_15m', 'ts_45m', '15STR_confirmed', '45STR_confirmed'],
      dtype='object')


# Add features
Key lv from 45STR_confirmed or 15STR_confirmed

In [3]:
import numpy as np

def add_keylv(df: pd.DataFrame) -> pd.DataFrame:
    n      = len(df)
    high   = df['high'].values
    low    = df['low'].values
    sig    = df['45STR_confirmed'].values

    keylv      = np.empty(n, dtype='float64')
    bars_since = np.zeros(n, dtype='int32')

    # seed: use first bar's low as initial key level
    keylv[0] = low[0]
    last_i   = 0  # index of last confirmed bar (start of look-back window)

    for i in range(1, n):
        if sig[i] == 1:
            keylv[i] = round(float(high[last_i : i].max()), 4)  # ← i+1 → i
            bars_since[i] = 0
            last_i = i
        elif sig[i] == -1:
            keylv[i] = round(float(low[last_i : i].min()), 4)   # ← i+1 → i
            bars_since[i] = 0
            last_i = i
        else:
            keylv[i]      = keylv[i - 1]
            bars_since[i] = bars_since[i - 1] + 1

    df = df.copy()
    df['45STR_keylv']    = keylv
    df['barsSince45STR'] = bars_since
    return df


df = add_keylv(df)
df.tail()


,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed,45STR_keylv,barsSince45STR
228767,1772667600000,72701.101562,72756.812500,72655.328125,72756.796875,56.436958,166.503159,1772667600000,0,0,0.000766,0.408642,0.000766,0.002288,1772667000000,1772666100000,0,0,74050.0,14
228768,1772667900000,72756.812500,72830.007812,72701.179688,72701.187500,43.030441,164.771942,1772667900000,0,0,-0.000765,0.314909,-0.000764,0.002266,1772667900000,1772666100000,0,0,74050.0,15
228769,1772668200000,72701.179688,72719.687500,72603.843750,72668.960938,169.471603,162.623138,1772668200000,1,1,-0.000443,1.262807,-0.000443,0.002238,1772667900000,1772666100000,0,0,74050.0,16
228770,1772668500000,72668.953125,72775.468750,72639.679688,72666.773438,275.874146,161.973770,1772668500000,1,1,-0.000030,2.120235,-0.000030,0.002229,1772667900000,1772666100000,0,0,74050.0,17
228771,1772668800000,72666.773438,72854.000000,72657.953125,72677.328125,168.545258,162.421127,1772668800000,0,0,0.000145,1.342385,0.000145,0.002235,1772668800000,1772668800000,0,0,74050.0,18


In [4]:
# create column

df["hDTK_45STR"]    = ((df["high"]  - df["45STR_keylv"]) / df["45STR_keylv"]).round(6)
df["lDTK_45STR"]   = ((df["low"]   - df["45STR_keylv"]) / df["45STR_keylv"]).round(6)
df["cDTK_45STR"] = ((df["close"] - df["45STR_keylv"]) / df["45STR_keylv"]).round(6)

df.drop(columns=["45STR_keylv"], inplace=True)
print(df.columns)
df.head()


Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'atr42', 'ts_5m',
       'label', 'train_mask', 'body_pct', 'vol_norm', 'close_ret1',
       'atr42_pct', 'ts_15m', 'ts_45m', '15STR_confirmed', '45STR_confirmed',
       'barsSince45STR', 'hDTK_45STR', 'lDTK_45STR', 'cDTK_45STR'],
      dtype='object')


,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask,...,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed,barsSince45STR,hDTK_45STR,lDTK_45STR,cDTK_45STR
0,1704037500000,42497.851562,42518.148438,42480.691406,42482.250000,56.512959,NaN,1704037500000,0,0,...,NaN,NaN,1704037500000,1704037500000,0,0,0,0.000882,0.000000,0.000037
1,1704037800000,42482.238281,42500.000000,42411.101562,42414.421875,47.354488,NaN,1704037800000,0,0,...,-0.001597,NaN,1704037500000,1704037500000,0,0,1,0.000455,-0.001638,-0.001560
2,1704038100000,42414.429688,42471.988281,42414.421875,42457.171875,43.228668,NaN,1704038100000,0,0,...,0.001008,NaN,1704037500000,1704037500000,0,0,2,-0.000205,-0.001560,-0.000554
3,1704038400000,42457.171875,42484.828125,42436.468750,42449.601562,74.418266,NaN,1704038400000,0,0,...,-0.000178,NaN,1704038400000,1704037500000,0,0,3,0.000097,-0.001041,-0.000732
4,1704038700000,42449.589844,42488.000000,42449.589844,42487.988281,38.818489,NaN,1704038700000,0,0,...,0.000904,NaN,1704038400000,1704037500000,0,0,4,0.000172,-0.000732,0.000172


# Analyse what to drop
send to 3analyseData to process

In [5]:
# cleaned data : ready for input
# df = df.drop(columns=["open", "high", "low", "close", "LTRkeylv", "ITRkeylv"])
# df.tail()

In [6]:
# save to JSONL for training
out_path = folder_path / "BTCUSD-5m-v6-augmented.jsonl"

df.to_json(out_path, orient="records", lines=True)
print(f"final shape : {df.shape}")
print(f"Saved {len(df)} rows to {out_path}")


final shape : (228772, 22)
Saved 228772 rows to /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-CandleScience/data/mlData/BTCUSD-5m-v6-augmented.jsonl
